In [ ]:
!pip install gradio langchain-openai langchain-community python-dotenv beautifulsoup4

In [ ]:
import os
import gradio as gr
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.documents import Document
from dotenv import load_dotenv

# Load environment variables from .env
load_dotenv()

In [ ]:
# Initialize the LLM (OpenRouter / Qwen)
llm = ChatOpenAI(
    model="qwen/qwen3-coder-next",  # or any OpenRouter model
    openai_api_base="https://openrouter.ai/api/v1",
    max_tokens=1000,
    temperature=0,
)

In [ ]:
# Setup Prompts and Chains
parser = StrOutputParser()

prompt_summary = PromptTemplate(
    template="Provide a very short, high-level summary of the following terms and conditions: {text}",
    input_variables=["text"]
)

prompt_offensive = PromptTemplate(
    template="Based on the following terms and conditions, list the top 7 most offensive or risky clauses. Format the output strictly as a bulleted list, where each bullet contains very very brief explanation of the clause: {text} \n ",
    input_variables=["text"]
)

summary_chain = prompt_summary | llm | parser
offensive_chain = prompt_offensive | llm | parser

In [ ]:
# Define analysis and chat helpers

def handle_input_change(method):
    if method == "URL Link":
        return gr.update(visible=True), gr.update(visible=False)
    else:
        return gr.update(visible=False), gr.update(visible=True)

def analyze(method, url, raw_text):
    text = ""
    if method == "URL Link":
        if not url:
            return "Please enter a URL.", "", [], {"text": "", "chat_history": []}
        try:
            loader = WebBaseLoader(url)
            documents = loader.load()
            text = documents[0].page_content
        except Exception as e:
            return f"Error loading URL: {e}", "", [], {"text": "", "chat_history": []}
    else:
        if not raw_text.strip():
            return "Please paste some text.", "", [], {"text": "", "chat_history": []}
        text = raw_text

    try:
        summary = summary_chain.invoke({"text": text})
        offensive = offensive_chain.invoke({"text": text})
        if not offensive or len(offensive.strip()) <= 5:
            offensive = "No highly offensive terms found."
            
        initial_history = [summary, offensive]
        state = {"text": text, "chat_history": initial_history}
        
        return summary, offensive, [], state
    except Exception as e:
        return f"Error analyzing: {e}", "", [], {"text": "", "chat_history": []}

def chat(user_message, chat_history, state):
    if not state or not state.get("text"):
        return "", chat_history + [("Please analyze a document first.", "")], state

    text = state["text"]
    history = state["chat_history"]

    # Format prompt to enforce scope
    prompt = f"System constraint: You are an AI legal assistant. You must ONLY answer questions directly related to the Terms and Conditions document. If the question is irrelevant, refuse to answer politely. User query: {user_message} (Answer briefly without jargon)"
    history.append(prompt)

    try:
        chat_result = llm.invoke(history)
        response = chat_result.content
        history.append(response)
        
        new_chat_history = chat_history + [(user_message, response)]
        state["chat_history"] = history
        
        return "", new_chat_history, state
    except Exception as e:
        return "", chat_history + [(user_message, f"Error: {e}")], state

In [ ]:
# Build Gradio Layout

with gr.Blocks(title="⚖️ Terms & Conditions Analyzer", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# ⚖️ Terms & Conditions Analyzer")
    gr.Markdown("Analyze Terms & Conditions via Web Scraping or Direct Paste, powered by Gradio.")
    
    # State tracker for session document text and model history
    state = gr.State({"text": "", "chat_history": []})
    
    with gr.Row():
        with gr.Column(scale=1):
            method_radio = gr.Radio(["URL Link", "Paste Text"], value="URL Link", label="Input Method")
            url_box = gr.Textbox(placeholder="Enter the URL of the Terms & Conditions...", label="URL Link")
            text_box = gr.Textbox(placeholder="Paste the Terms & Conditions text here...", label="Paste Text", visible=False, lines=8)
            
            analyze_btn = gr.Button("Analyze Document", variant="primary")
            
        with gr.Column(scale=1):
            summary_out = gr.Textbox(label="📝 Summary", interactive=False, lines=4)
            offensive_out = gr.Textbox(label="🚩 Offensive Terms", interactive=False, lines=6)
            
    method_radio.change(handle_input_change, inputs=[method_radio], outputs=[url_box, text_box])
    
    gr.HTML("<hr>")
    gr.Markdown("### 💬 Chat with the Document")
    
    chatbot = gr.Chatbot(label="Chat History")
    with gr.Row():
        msg_input = gr.Textbox(placeholder="Ask a question about the Terms & Conditions...", scale=9)
        send_btn = gr.Button("Send", scale=1)
        
    # Set button and submit actions
    analyze_btn.click(
        analyze,
        inputs=[method_radio, url_box, text_box],
        outputs=[summary_out, offensive_out, chatbot, state]
    )
    
    send_btn.click(
        chat,
        inputs=[msg_input, chatbot, state],
        outputs=[msg_input, chatbot, state]
    )
    msg_input.submit(
        chat,
        inputs=[msg_input, chatbot, state],
        outputs=[msg_input, chatbot, state]
    )

# Launch the inline dashboard
demo.launch(inline=True)